In [ ]:
pip install pandas openpyxl

In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
from openai import OpenAI

MODEL = "openai/gpt-5.2"          # OpenRouter model identifier
MAX_OUTPUT_TOKENS = 10000
TEMPERATURE = 0.1
TOP_P = 1

INPUT_EXCEL = "Problems_for_ experiment.xlsx"
OUTPUT_CSV = "generated_diagrams.csv"
PUML_FOLDER = "puml_outputs"
SCENARIO_ID_COL = "ScenarioID"
SCENARIO_TEXT_COL = "ScenarioDescription"

os.makedirs(PUML_FOLDER, exist_ok=True)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY environment variable not set.")

In [ ]:
def trim_to_enduml(text: str) -> str:
    if "@enduml" in text:
        return text.split("@enduml")[0] + "@enduml"
    return text


def build_prompt(requirements: str) -> str:
    return f"""
You are an expert software engineer specialized in UML modeling and requirements analysis.

Your task is to generate a syntactically correct and semantically complete PlantUML activity diagram from the given software requirements.

###Constraints:
1. Output ONLY valid PlantUML code between @startuml and @enduml.
2. For single-user requirements, generate a flat activity diagram.
3. For multi-user requirements, use swimlanes (|Actor|) to separate activities.
4. Preserve the logical flow of the actions as mentioned in requirements. 
5. If the requirement has alternative flows, model them using decisions as diamond nodes using `if (...) then (yes/no)` syntax with conditions and two logical outcomes.
6. Represent loops, iterations, and repetitions explicitly.
7. Use `fork` / `fork again` / `end fork` for concurrent or parallel activities.
8. Use short verb–object labels (e.g., "Validate payment").
9. First swimlane must be added after @startuml and before 'start'.
10. Avoid duplicate or redundant action nodes.
11. Do not invent activities that are not supported by the requirements.
12. Every activity in the diagram must be traceable to the requirement.
13. Do not provide explanations or reasoning. Return only valid PlantUML code.


###Example 1:

Requirements:
"A customer browses products and adds items to the cart. The system checks item availability. If available, the customer proceeds to checkout and makes payment. If payment is successful, the order is confirmed. Otherwise, the customer is asked to retry."

PlantUML Output:
@startuml
|Customer|
start
:Browse Products;
:Add Item to Cart;
|System|
if (Item Available?) then (yes)
  |Customer|
  :Proceed to Checkout;
  :Make Payment;
  |System|
  if (Payment Successful?) then (yes)
    :Confirm Order;
  else (no)
    |Customer|
    :Retry Payment;
  endif
else (no)
  :Notify Unavailability;
endif
stop
@enduml

###Example 2:

Requirements:
"A student logs into the university portal and selects a course to register. The system checks if the student meets the prerequisites. If prerequisites are not met, the student is notified and registration is denied. If prerequisites are met, the system checks seat availability. If no seats are available, the student is added to a waitlist. If seats are available, the system checks for a timetable conflict with already registered courses. If a conflict exists, the student is notified and asked to select another course. If no conflict exists, the student confirms registration. The system registers the student, deducts one seat, sends a confirmation email, and updates the academic record. The registrar can at any time log in and view all registrations, approve or reject manual registration requests, and generate a registration report."

PlantUML Output:
@startuml

|Student|
start
:Enter username and password;
|System|
if (Credentials valid?) then (yes)
  |Student|
  :Select course to register;
  |System|
  if (Prerequisites met?) then (no)
    :Notify student - prerequisites not met;
    |Student|
    :Registration denied;
  else (yes)
    if (Seats available?) then (no)
      :Add student to waitlist;
      :Notify student - added to waitlist;
    else (yes)
      if (Timetable conflict?) then (yes)
        :Notify student - timetable conflict;
        |Student|
        :Select another course;
      else (no)
        |Student|
        :Confirm registration;
        |System|
        :Register student;
        :Deduct one seat;
        :Send confirmation email;
        :Update academic record;
      endif
    endif
  endif
else (no)
  :Notify student - invalid credentials;
endif

|Registrar|
:Login to portal;
|System|
if (Registrar credentials valid?) then (yes)
  |Registrar|
  :Select operation;
  if (View registrations?) then (yes)
    |System|
    :Display all registrations;
  else (no)
    if (Manual registration request?) then (yes)
      |System|
      :Display pending requests;
      |Registrar|
      if (Approve request?) then (yes)
        |System|
        :Approve and register student;
      else (no)
        :Reject request;
        :Notify student of rejection;
      endif
    else (no)
      :Generate registration report;
    endif
  endif
else (no)
  :Notify registrar - invalid credentials;
endif

stop
@enduml

###Requirements:
{requirements}

###Output:
One PlantUML code block only (from '@startuml' to '@enduml')

"""

In [ ]:
def generate_activity_diagram(prompt: str):
    response = client.chat.completions.create(  
        model=MODEL,
        messages=[{"role": "user", "content": prompt}], 
        max_tokens=MAX_OUTPUT_TOKENS,           
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    text = response.choices[0].message.content or ""  
    text = trim_to_enduml(text)

    usage = getattr(response, "usage", None)
    usage_info = {
        "input_tokens": getattr(usage, "prompt_tokens", None),     
        "output_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "timestamp": datetime.now().isoformat(),
        "model": MODEL
    }

    return text, usage_info

In [ ]:
# =========================
# Load Excel
# =========================
df = pd.read_excel(INPUT_EXCEL)

results = []

print(f"Total scenarios found: {len(df)}")

for idx, row in df.iterrows():
    scenario_id = str(row[SCENARIO_ID_COL])
    scenario_text = str(row[SCENARIO_TEXT_COL])

    print(f"\nProcessing Scenario: {scenario_id}")

    try:
        prompt = build_prompt(scenario_text)
        output_text, usage = generate_activity_diagram(prompt)

        # =========================
        # Save .puml file
        # =========================
        puml_filename = f"{scenario_id}.puml"
        puml_path = os.path.join(PUML_FOLDER, puml_filename)

        with open(puml_path, "w", encoding="utf-8") as f:
            f.write(output_text)

        # =========================
        # Store results for CSV
        # =========================
        results.append({
            "ScenarioID": scenario_id,
            "ScenarioDescription": scenario_text,
            "GeneratedPlantUML": output_text,
            "InputTokens": usage["input_tokens"],
            "OutputTokens": usage["output_tokens"],
            "TotalTokens": usage["total_tokens"],
            "Model": usage["model"],
            "Timestamp": usage["timestamp"],
            "PUML_File": puml_path
        })

        # small delay (good practice for batches)
        time.sleep(0.5)

    except Exception as e:
        print(f" Error in Scenario {scenario_id}: {e}")

        results.append({
            "ScenarioID": scenario_id,
            "ScenarioDescription": scenario_text,
            "GeneratedPlantUML": "ERROR",
            "InputTokens": None,
            "OutputTokens": None,
            "TotalTokens": None,
            "Model": MODEL,
            "Timestamp": datetime.now().isoformat(),
            "PUML_File": None
        })

# =========================
# Save CSV
# =========================
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV, index=False)

print("\n Processing complete.")
print(f"CSV saved to: {OUTPUT_CSV}")
print(f"PUML files folder: {PUML_FOLDER}")